# 05 — PhishTank API Extraction

**C1 Source Type:** `Service web (API REST)`

---

## Objective

Extract French-targeted phishing URLs from the [PhishTank](https://www.phishtank.com/) open database.
PhishTank is a community-driven anti-phishing site operated by Cisco/OpenDNS that maintains a real-time
feed of **verified phishing URLs** submitted by the community.

### Why PhishTank?

| Criterion | Detail |
|-----------|--------|
| Format | REST API (JSON) + downloadable CSV/JSON dump |
| Coverage | 100K+ verified phishing URLs, global |
| French signal | `.fr` domains + French brand impersonation (URSSAF, Ameli, CAF, La Poste, banks) |
| C1 requirement | Satisfies *"service web / API REST"* extraction type |
| Cost | Free (API key required) |

### Pipeline

```
PhishTank feed (JSON) → Filter French targets → Deduplicate → Export CSV
```

### Output

- `data/raw/api/phishtank/phishtank_french_<N>_<date>.csv`

In [ ]:
# ── Imports & Constants ──────────────────────────────────────────────
from __future__ import annotations

import hashlib
import re
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlparse

import httpx
import pandas as pd
from dotenv import load_dotenv

# ── Load .env ────────────────────────────────────────────────────────
ENV_PATH: Path = Path("data/.env")
load_dotenv(ENV_PATH)

# ── Configuration ────────────────────────────────────────────────────
# PhishTank provides a downloadable JSON feed of all verified phishing URLs.
# The online-valid.json endpoint does NOT require an API key.
PHISHTANK_FEED_URL: str = "http://data.phishtank.com/data/online-valid.json"

OUTPUT_DIR: Path = Path("data/raw/api/phishtank")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# French-targeted filtering
# 1. Domains ending in .fr / .gouv.fr / .asso.fr
# 2. URLs containing French brand keywords
FR_TLD_PATTERN: re.Pattern = re.compile(r"\.fr(/|$|:)", re.IGNORECASE)

FR_BRAND_KEYWORDS: list[str] = [
    # Government / health
    "urssaf", "ameli", "impots", "dgfip", "caf", "cpam", "securite-sociale",
    "france-connect", "franceconnect", "service-public", "gouv",
    # Postal / delivery
    "laposte", "la-poste", "colissimo", "chronopost", "mondial-relay",
    # Banking
    "credit-agricole", "creditagricole", "bnp", "bnpparibas",
    "banque-postale", "banquepostale", "societe-generale", "societegenerale",
    "lcl", "caisse-epargne", "credit-mutuel",
    # Telecom
    "orange", "sfr", "bouygues", "free",
    # E-commerce
    "leboncoin", "cdiscount", "fnac",
]

print(f"Feed URL    : {PHISHTANK_FEED_URL}")
print(f"Output dir  : {OUTPUT_DIR.resolve()}")
print("FR TLDs     : .fr, .gouv.fr, .asso.fr")
print(f"FR keywords : {len(FR_BRAND_KEYWORDS)} brand patterns")

Feed URL    : http://data.phishtank.com/data/online-valid.json
Output dir  : /Users/michaeladebayo/Documents/Simplon/brief_projects/sicurre/data/raw/api/phishtank
FR TLDs     : .fr, .gouv.fr, .asso.fr
FR keywords : 34 brand patterns


## 1. Download PhishTank Feed

PhishTank publishes `online-valid.json` — a full dump of all **currently verified** phishing URLs.
This is updated roughly every hour. File size is typically 20–60 MB.

> **Note:** PhishTank rate-limits to ~1 request/day per device/network. If you get
> HTTP 429, wait 24h or use the pre-downloaded CSV dump in `data/raw/api/phishtank/`.

In [ ]:
# ── Download the full PhishTank feed ─────────────────────────────────
# The feed can be large (20-60 MB), so we set a generous timeout.
# PhishTank rate-limits to ~1 request/day per device/network.

print("Downloading PhishTank feed (this may take 30-60s)...")

headers: dict[str, str] = {
    "User-Agent": "phishtank/sicurre-research",
}

with httpx.Client(timeout=120.0, follow_redirects=True) as client:
    response = client.get(PHISHTANK_FEED_URL, headers=headers)
    response.raise_for_status()

phishtank_data: list[dict] = response.json()

print(f"Downloaded  : {len(phishtank_data):,} verified phishing entries")
print(f"Sample keys : {list(phishtank_data[0].keys()) if phishtank_data else 'N/A'}")

In [ ]:
# ── Preview raw data ─────────────────────────────────────────────────
df_raw: pd.DataFrame = pd.DataFrame(phishtank_data)
print(f"Shape: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")
df_raw.head(3)

## 2. Filter French-Targeted URLs

We apply two complementary filters:

1. **TLD filter:** URLs with `.fr` domain (or subdomains like `.gouv.fr`, `.asso.fr`)
2. **Brand filter:** URLs containing French brand keywords (URSSAF, Ameli, BNP, etc.)

The union of both filters gives us the French-targeted subset.

In [ ]:
def is_french_target(url: str) -> tuple[bool, str]:
    """Check if a URL targets French users. Returns (match, reason)."""
    url_lower: str = url.lower()
    
    # Check .fr TLD
    if FR_TLD_PATTERN.search(url_lower):
        return True, "fr_tld"
    
    # Check French brand keywords
    for keyword in FR_BRAND_KEYWORDS:
        if keyword in url_lower:
            return True, f"brand:{keyword}"
    
    return False, ""


# ── Apply filter ──────────────────────────────────────────────────────
results: list[dict] = []
for entry in phishtank_data:
    url: str = entry.get("url", "")
    match, reason = is_french_target(url)
    if match:
        parsed = urlparse(url)
        results.append({
            "url": url,
            "domain": parsed.netloc,
            "phish_id": entry.get("phish_id", ""),
            "submission_time": entry.get("submission_time", ""),
            "verified": entry.get("verified", ""),
            "verified_time": entry.get("verification_time", ""),
            "target": entry.get("target", ""),
            "filter_reason": reason,
            "label": "phishing",
            "source": "phishtank_api",
        })

df_french: pd.DataFrame = pd.DataFrame(results)
print(f"French-targeted : {len(df_french):,} / {len(phishtank_data):,} ({len(df_french)/len(phishtank_data)*100:.1f}%)")

if not df_french.empty:
    print(f"\nFilter breakdown:")
    print(df_french["filter_reason"].value_counts().head(15))

In [ ]:
# ── Preview filtered results ──────────────────────────────────────────
if not df_french.empty:
    print(f"Unique domains: {df_french['domain'].nunique()}")
    print(f"\nTop 15 targeted domains:")
    print(df_french["domain"].value_counts().head(15))
    print(f"\nTop targets (brand names):")
    print(df_french["target"].value_counts().head(10))
    df_french.head(5)
else:
    print("No French-targeted URLs found in current feed.")
    print("This can happen if the feed is stale or French entries have been cleaned.")

## 3. Deduplicate & Clean

In [ ]:
# ── Deduplication by URL hash ─────────────────────────────────────────
if not df_french.empty:
    before: int = len(df_french)
    
    # Normalize URLs (strip trailing slash, lowercase)
    df_french["url_normalized"] = df_french["url"].str.lower().str.rstrip("/")
    df_french["url_hash"] = df_french["url_normalized"].apply(
        lambda u: hashlib.sha256(u.encode()).hexdigest()
    )
    
    # Keep first occurrence (earliest submission)
    df_dedup: pd.DataFrame = (
        df_french
        .sort_values("submission_time")
        .drop_duplicates(subset="url_hash", keep="first")
        .drop(columns=["url_normalized", "url_hash"])
        .reset_index(drop=True)
    )
    
    after: int = len(df_dedup)
    print(f"Before dedup : {before:,}")
    print(f"After dedup  : {after:,}")
    print(f"Removed      : {before - after:,} ({(before - after) / before * 100:.1f}%)")
else:
    df_dedup = df_french

## 4. Export

In [ ]:
# ── Export to CSV ─────────────────────────────────────────────────────
timestamp: str = datetime.now(timezone.utc).strftime("%Y%m%d")
n_rows: int = len(df_dedup)

if n_rows > 0:
    filename: str = f"phishtank_french_{n_rows}_{timestamp}.csv"
    output_path: Path = OUTPUT_DIR / filename
    
    df_dedup.to_csv(output_path, index=False, encoding="utf-8")
    
    size_mb: float = output_path.stat().st_size / (1024 * 1024)
    print(f"Exported     : {output_path}")
    print(f"Rows         : {n_rows:,}")
    print(f"Size         : {size_mb:.2f} MB")
    print(f"Columns      : {list(df_dedup.columns)}")
else:
    print("Nothing to export — 0 French-targeted URLs in this feed snapshot.")
    print("Try again later or download the full historical archive from phishtank.com.")

## 5. Summary

### What this notebook demonstrates (C1)

| Criterion | Evidence |
|-----------|----------|
| **Source type** | REST API (JSON feed) |
| **Automation** | Programmatic download + filtering |
| **Domain filtering** | `.fr` TLD + 30+ French brand keywords |
| **Deduplication** | SHA-256 URL hashing |
| **Output** | Structured CSV with provenance columns |

### Limitations

- PhishTank feed contains **URLs only** — no email bodies. These URLs are used for:
  - URL reputation features in the phishing classifier
  - Training a URL-based scoring model alongside CamemBERTv2 text classification
  - Cross-referencing with CERT-FR IOCs
- French-targeted entries are a small fraction (~1-3%) of the global feed
- Feed is a point-in-time snapshot — run periodically for longitudinal data
- Rate-limited to ~1 request/day per device/network (HTTP 429 if exceeded)